# PRC / CARR — Smoke Test (Free Colab)

**Purpose:** Verify the code runs correctly before full training on AWS EC2.  
**This runs:** Experiment 1 only · MovieLens-1M only · 3 epochs (not 100)  
**Expected runtime:** ~10–20 minutes on a free T4 GPU

---
**Before running:** Go to `Runtime → Change runtime type → T4 GPU`

## Step 1 — Check GPU

In [ ]:
import torch

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("No GPU — go to Runtime > Change runtime type > T4 GPU")

## Step 2 — Clone Repo from GitHub

In [ ]:
import os

if not os.path.exists('/content/progressive_reasoning_collapse'):
    !git clone https://github.com/jkinarthur/progressive_reasoning_collapse.git
else:
    !cd /content/progressive_reasoning_collapse && git pull

%cd /content/progressive_reasoning_collapse/experiments
print("Working directory:", os.getcwd())
print("Files:", os.listdir('.'))

## Step 3 — Install Requirements

In [ ]:
!pip install -q -r requirements.txt
print("Dependencies installed.")

## Step 4 — Patch Config for Smoke Test
Reduces epochs from 100 → 3 so it runs quickly on free Colab.

In [ ]:
config_path = '/content/progressive_reasoning_collapse/experiments/config.py'

with open(config_path, 'r') as f:
    content = f.read()

# Patch epochs and batch size for smoke test
patched = content.replace('num_epochs: int = 100', 'num_epochs: int = 3')
patched = patched.replace('batch_size: int = 64', 'batch_size: int = 32')
patched = patched.replace('num_runs: int = 5', 'num_runs: int = 1')

with open(config_path, 'w') as f:
    f.write(patched)

print("Config patched: epochs=3, batch_size=32, num_runs=1")
print("NOTE: This is for smoke testing only — full runs need AWS EC2")

## Step 5 — Run Experiment 1 (Smoke Test)

In [ ]:
!python main.py --experiments 1 --datasets ml-1m --device cuda

## Step 6 — Check Outputs

In [ ]:
import os
from pathlib import Path

results_dir = Path('/content/progressive_reasoning_collapse/results')
figures_dir = Path('/content/progressive_reasoning_collapse/figures')

print("=== Results ===")
for f in results_dir.rglob('*'):
    print(f.relative_to(results_dir))

print("\n=== Figures ===")
for f in figures_dir.rglob('*'):
    print(f.relative_to(figures_dir))

## Step 7 — Display Generated Figures

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from pathlib import Path

figures = list(Path('/content/progressive_reasoning_collapse/figures').rglob('*.png'))
figures += list(Path('/content/progressive_reasoning_collapse/figures').rglob('*.pdf'))

if figures:
    for fig_path in figures[:4]:  # Show first 4
        if fig_path.suffix == '.png':
            img = mpimg.imread(str(fig_path))
            plt.figure(figsize=(10, 6))
            plt.imshow(img)
            plt.title(fig_path.name)
            plt.axis('off')
            plt.show()
else:
    print("No figures yet — experiment may still be running or check logs above")

## Step 8 — Download Results to Your PC
Run this to zip and download all results.

In [ ]:
import shutil
from google.colab import files

# Zip results + figures
shutil.make_archive('/content/smoke_test_results', 'zip', '/content/progressive_reasoning_collapse', 'results')
files.download('/content/smoke_test_results.zip')
print("Download started.")

---
## Smoke Test Complete

If Experiment 1 ran without errors above, your code is **ready for full training on AWS EC2**.

**Next step:** Once your AWS quota increase is approved, run:
```bash
docker run --gpus all \
  -v $(pwd)/results:/workspace/results \
  carr-experiment \
  python main.py --experiments all --datasets ml-1m amazon-beauty
```